# Audited Graph WaveNet — Final64

Notebook ini digunakan untuk membangun dan mengevaluasi Graph WaveNet pada dataset Final64.

## Tujuan

Menguji tiga konfigurasi graph dengan backbone Graph WaveNet yang sama:

1. **Graph WaveNet Fixed**
   - menggunakan geographic graph Final64.

2. **Graph WaveNet Adaptive**
   - menggunakan adjacency matrix yang dipelajari selama training.

3. **Graph WaveNet Hybrid**
   - menggunakan geographic graph dan learned adaptive graph secara bersamaan.

## Adaptasi Final64

Dataset memiliki:

- 64 koridor
- 12 timestep historis
- 1 fitur `current_speed`
- target t+1, t+3, dan t+6

Seluruh model menggunakan train, validation, test split dan evaluation mask yang sama.

## Screening

Eksperimen awal menggunakan seed 42.

Model yang menjanjikan kemudian dapat dilanjutkan ke eksperimen multi-seed.

## 1. Setup Environment

### Tujuan

Memuat library yang diperlukan dan menemukan root repository secara otomatis.

### Output

Cell menghasilkan:

- `ROOT`
- `DATA_DIR`
- versi TensorFlow
- lokasi repository yang digunakan

In [1]:
# ============================================================
# SETUP ENVIRONMENT
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# FIND REPOSITORY ROOT
# ============================================================

cwd = Path.cwd()

ROOT = None

for path in [cwd] + list(cwd.parents):
    if (path / "data" / "final_64").exists():
        ROOT = path
        break


if ROOT is None:
    raise FileNotFoundError("Root repository dengan data/final_64 tidak ditemukan.")


DATA_DIR = ROOT / "data" / "final_64"


# ============================================================
# BASIC CHECK
# ============================================================

print("TensorFlow :", tf.__version__)
print("Current dir:", cwd)
print("ROOT       :", ROOT)
print("DATA_DIR   :", DATA_DIR)
print("Exists     :", DATA_DIR.exists())

TensorFlow : 2.21.0
Current dir: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/notebooks/notebooks
ROOT       : /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent
DATA_DIR   : /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/data/final_64
Exists     : True


## 2. Load Final64 Dataset

### Tujuan

Memuat train, validation, test, serta scaler yang sudah digunakan pada eksperimen sebelumnya.

### Input

File:

- `stgnn_train.npz`
- `stgnn_val.npz`
- `stgnn_test.npz`
- `stgnn_scaler.npz`

### Output

Cell menghasilkan:

- `X_train`, `y_train`, `mask_train`
- `X_val`, `y_val`, `mask_val`
- `X_test`, `y_test`, `mask_test`
- `SCALER_MEAN`
- `SCALER_SCALE`

Format utama:

- `X = (sample, 12, 64, 1)`
- `y = (sample, 3, 64)`

In [2]:
# ============================================================
# LOAD FINAL64 DATASET
# ============================================================

train_npz = np.load(DATA_DIR / "stgnn_train.npz", allow_pickle=True)

val_npz = np.load(DATA_DIR / "stgnn_val.npz", allow_pickle=True)

test_npz = np.load(DATA_DIR / "stgnn_test.npz", allow_pickle=True)

scaler_npz = np.load(DATA_DIR / "stgnn_scaler.npz", allow_pickle=True)


# ============================================================
# EXTRACT ARRAYS
# ============================================================

X_train = train_npz["X"].astype(np.float32)
y_train = train_npz["y"].astype(np.float32)
mask_train = train_npz["mask"].astype(np.float32)

X_val = val_npz["X"].astype(np.float32)
y_val = val_npz["y"].astype(np.float32)
mask_val = val_npz["mask"].astype(np.float32)

X_test = test_npz["X"].astype(np.float32)
y_test = test_npz["y"].astype(np.float32)
mask_test = test_npz["mask"].astype(np.float32)


# ============================================================
# SCALER
# ============================================================

SCALER_MEAN = float(np.asarray(scaler_npz["mean"]).reshape(-1)[0])

SCALER_SCALE = float(np.asarray(scaler_npz["scale"]).reshape(-1)[0])


# ============================================================
# BASIC CONFIG
# ============================================================

NUM_NODES = X_train.shape[2]
INPUT_LENGTH = X_train.shape[1]
NUM_HORIZONS = y_train.shape[1]

HORIZONS = [1, 3, 6]


# ============================================================
# CHECK
# ============================================================

print("=== FINAL64 DATASET ===")

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("mask_train :", mask_train.shape)

print()

print("X_val   :", X_val.shape)
print("y_val   :", y_val.shape)
print("mask_val:", mask_val.shape)

print()

print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)
print("mask_test:", mask_test.shape)

print()

print("NUM_NODES    :", NUM_NODES)
print("INPUT_LENGTH :", INPUT_LENGTH)
print("HORIZONS     :", HORIZONS)

print()

print("Scaler mean  :", SCALER_MEAN)
print("Scaler scale :", SCALER_SCALE)

print()

print(
    "Valid target points:",
    {"train": int(mask_train.sum()), "val": int(mask_val.sum()), "test": int(mask_test.sum())},
)

=== FINAL64 DATASET ===
X_train : (415, 12, 64, 1)
y_train : (415, 3, 64)
mask_train : (415, 3, 64)

X_val   : (92, 12, 64, 1)
y_val   : (92, 3, 64)
mask_val: (92, 3, 64)

X_test  : (93, 12, 64, 1)
y_test  : (93, 3, 64)
mask_test: (93, 3, 64)

NUM_NODES    : 64
INPUT_LENGTH : 12
HORIZONS     : [1, 3, 6]

Scaler mean  : 30.50661914460285
Scaler scale : 11.331123867533398

Valid target points: {'train': 44736, 'val': 9024, 'test': 8896}


## 3. Load Fixed Geographic Graph

### Tujuan

Memuat geographic graph Final64 yang sudah dibangun dari kedekatan geografis antarkoridor.

Untuk Graph WaveNet, fixed graph akan direpresentasikan sebagai dua directional supports:

1. forward transition
2. reverse transition

Kedua support dibangun dari `directed_knn_adjacency` dan dinormalisasi per baris.

### Output

Cell menghasilkan:

- `A_directed`
- `A_forward`
- `A_reverse`
- `FIXED_SUPPORTS`

In [3]:
# ============================================================
# LOAD FIXED GRAPH
# ============================================================

FIXED_GRAPH_FILE = ROOT / "outputs" / "final_64" / "fixed_graph" / "fixed_graph_64.npz"

graph_npz = np.load(FIXED_GRAPH_FILE, allow_pickle=True)

print("Graph keys:")
print(graph_npz.files)


# ============================================================
# DIRECTED KNN ADJACENCY
# ============================================================

A_directed = graph_npz["directed_knn_adjacency"].astype(np.float32)


# ============================================================
# ROW NORMALIZATION
# D^-1 A
# ============================================================


def row_normalize_adjacency(adj):

    adj = np.asarray(adj, dtype=np.float32)

    degree = adj.sum(axis=1)

    degree_inv = np.zeros_like(degree, dtype=np.float32)

    nonzero = degree > 0

    degree_inv[nonzero] = 1.0 / degree[nonzero]

    normalized = degree_inv[:, None] * adj

    return normalized.astype(np.float32)


# ============================================================
# FORWARD + REVERSE TRANSITION
# ============================================================

A_forward = row_normalize_adjacency(A_directed)

A_reverse = row_normalize_adjacency(A_directed.T)

FIXED_SUPPORTS = [A_forward, A_reverse]


# ============================================================
# CHECK
# ============================================================

print("\n=== FIXED GRAPH ===")

print("A_directed:", A_directed.shape)

print("A_forward :", A_forward.shape)

print("A_reverse :", A_reverse.shape)

print("Directed edges:", int(A_directed.sum()))

print("All finite:", np.isfinite(A_forward).all() and np.isfinite(A_reverse).all())

print("\nForward row sums first 10:")

print(np.round(A_forward.sum(axis=1)[:10], 4))

print("\nReverse row sums first 10:")

print(np.round(A_reverse.sum(axis=1)[:10], 4))

Graph keys:
['adjacency', 'adjacency_with_self', 'normalized_adjacency', 'distance_matrix', 'directed_knn_adjacency']

=== FIXED GRAPH ===
A_directed: (64, 64)
A_forward : (64, 64)
A_reverse : (64, 64)
Directed edges: 256
All finite: True

Forward row sums first 10:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Reverse row sums first 10:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 4. Masked Loss and Evaluation

### Tujuan

Dataset Final64 memiliki target yang tidak semuanya valid pada setiap horizon.

Karena itu, training dan evaluasi hanya dilakukan pada target dengan `mask = 1`.

### Training

Loss menggunakan **Masked Mean Squared Error (MSE)** pada standardized scale.

### Evaluation

Prediksi dikembalikan ke satuan kecepatan asli menggunakan scaler, kemudian dihitung:

- MAE
- RMSE

secara terpisah untuk:

- t+1
- t+3
- t+6

In [4]:
# ============================================================
# COMBINE TARGET + MASK
# ============================================================


def combine_y_mask(y, mask):
    """
    Output shape:
    (batch, horizon, node, 2)

    channel 0 = target
    channel 1 = mask
    """

    return np.stack([y, mask], axis=-1).astype(np.float32)


y_train_masked = combine_y_mask(y_train, mask_train)

y_val_masked = combine_y_mask(y_val, mask_val)


# ============================================================
# MASKED MSE LOSS
# ============================================================


@tf.keras.utils.register_keras_serializable()
def masked_mse(y_true_with_mask, y_pred):

    y_true = y_true_with_mask[..., 0]
    mask = y_true_with_mask[..., 1]

    squared_error = tf.square(y_true - y_pred)

    masked_error = squared_error * mask

    valid_count = tf.maximum(tf.reduce_sum(mask), 1.0)

    return tf.reduce_sum(masked_error) / valid_count


# ============================================================
# INVERSE SCALER
# ============================================================


def inverse_scale(x):

    return x * SCALER_SCALE + SCALER_MEAN


# ============================================================
# EVALUATION
# ============================================================


def evaluate_predictions(y_true_std, y_pred_std, mask, model_name="Model"):

    y_true_real = inverse_scale(y_true_std)

    y_pred_real = inverse_scale(y_pred_std)

    rows = []

    for h_idx, horizon in enumerate(HORIZONS):
        valid = mask[:, h_idx, :] > 0

        actual = y_true_real[:, h_idx, :][valid]

        pred = y_pred_real[:, h_idx, :][valid]

        mae = mean_absolute_error(actual, pred)

        rmse = np.sqrt(mean_squared_error(actual, pred))

        rows.append(
            {
                "model": model_name,
                "horizon": f"t+{horizon}",
                "MAE": float(mae),
                "RMSE": float(rmse),
                "N": int(valid.sum()),
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# CALLBACKS
# ============================================================


def make_callbacks():

    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
        ),
    ]


# ============================================================
# CHECK
# ============================================================

print("y_train_masked:", y_train_masked.shape)

print("y_val_masked  :", y_val_masked.shape)

print(
    "Test valid per horizon:",
    {f"t+{h}": int(mask_test[:, i, :].sum()) for i, h in enumerate(HORIZONS)},
)

y_train_masked: (415, 3, 64, 2)
y_val_masked  : (92, 3, 64, 2)
Test valid per horizon: {'t+1': 2944, 't+3': 2944, 't+6': 3008}


## 5. Graph WaveNet Configuration

### Tujuan

Menentukan konfigurasi utama Graph WaveNet yang mengikuti struktur implementasi official.

### Konfigurasi

Model menggunakan:

- 4 blocks
- 2 layers per block
- kernel size 2
- dilation pattern 1, 2 pada setiap block
- residual channels 32
- dilation channels 32
- skip channels 256
- end channels 512
- graph diffusion order 2
- adaptive embedding dimension 10
- dropout 0.3

### Receptive Field

Dengan 4 blocks, 2 layers per block, kernel size 2, dan dilation 1 lalu 2 di setiap block, receptive field menjadi:

`13 timestep`

Karena input Final64 memiliki 12 timestep, input akan di-left-pad 1 timestep agar sesuai receptive field model.

In [5]:
# ============================================================
# GRAPH WAVENET CONFIG
# ============================================================

GW_BLOCKS = 4
GW_LAYERS = 2

GW_KERNEL_SIZE = 2

GW_RESIDUAL_CHANNELS = 32
GW_DILATION_CHANNELS = 32

GW_SKIP_CHANNELS = 256
GW_END_CHANNELS = 512

GW_DIFFUSION_ORDER = 2

GW_ADAPTIVE_EMBED_DIM = 10

GW_DROPOUT = 0.3

GW_LEARNING_RATE = 1e-3

GW_BATCH_SIZE = 32

GW_MAX_EPOCHS = 300


# ============================================================
# RECEPTIVE FIELD
# ============================================================

receptive_field = 1

for _ in range(GW_BLOCKS):
    additional_scope = GW_KERNEL_SIZE - 1

    dilation = 1

    for _ in range(GW_LAYERS):
        receptive_field += additional_scope

        dilation *= 2

        additional_scope *= 2


GW_RECEPTIVE_FIELD = receptive_field


# ============================================================
# CHECK
# ============================================================

print("=== GRAPH WAVENET CONFIG ===")

print("Blocks              :", GW_BLOCKS)

print("Layers / block      :", GW_LAYERS)

print("Total temporal layer:", GW_BLOCKS * GW_LAYERS)

print("Kernel size         :", GW_KERNEL_SIZE)

print("Residual channels   :", GW_RESIDUAL_CHANNELS)

print("Dilation channels   :", GW_DILATION_CHANNELS)

print("Skip channels       :", GW_SKIP_CHANNELS)

print("End channels        :", GW_END_CHANNELS)

print("Diffusion order     :", GW_DIFFUSION_ORDER)

print("Adaptive emb dim    :", GW_ADAPTIVE_EMBED_DIM)

print("Dropout             :", GW_DROPOUT)

print("Receptive field     :", GW_RECEPTIVE_FIELD)

print("Input length        :", INPUT_LENGTH)

print("Required left pad   :", max(GW_RECEPTIVE_FIELD - INPUT_LENGTH, 0))

=== GRAPH WAVENET CONFIG ===
Blocks              : 4
Layers / block      : 2
Total temporal layer: 8
Kernel size         : 2
Residual channels   : 32
Dilation channels   : 32
Skip channels       : 256
End channels        : 512
Diffusion order     : 2
Adaptive emb dim    : 10
Dropout             : 0.3
Receptive field     : 13
Input length        : 12
Required left pad   : 1


## 6. Diffusion Graph Convolution

### Tujuan

Membuat operasi graph convolution yang digunakan Graph WaveNet.

Untuk setiap graph support, informasi node dipropagasikan sampai diffusion order 2:

- orde 0: fitur node asli
- orde 1: informasi dari tetangga satu langkah
- orde 2: informasi dari propagasi dua langkah

Semua hasil diffusion kemudian digabung pada dimensi fitur dan diproyeksikan kembali ke jumlah channel yang diinginkan.

Layer ini dapat digunakan untuk:

- fixed geographic supports
- adaptive support
- kombinasi fixed + adaptive supports

In [6]:
# ============================================================
# GRAPH PROPAGATION
# ============================================================


def graph_propagate(x, support):
    """
    x:
        (batch, time, nodes, channels)

    support:
        (nodes, nodes)

    output:
        (batch, time, nodes, channels)
    """

    return tf.einsum("btnc,nm->btmc", x, support)


# ============================================================
# DIFFUSION GRAPH CONVOLUTION
# ============================================================


class DiffusionGraphConv(tf.keras.layers.Layer):
    def __init__(self, output_channels, diffusion_order=2, dropout=0.0, **kwargs):

        super().__init__(**kwargs)

        self.output_channels = output_channels

        self.diffusion_order = diffusion_order

        self.dropout_rate = dropout

        self.projection = tf.keras.layers.Dense(output_channels)

        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, supports, training=False):

        # --------------------------------------------
        # Order 0 = original features
        # --------------------------------------------

        diffusion_outputs = [x]

        # --------------------------------------------
        # Diffusion for every support
        # --------------------------------------------

        for support in supports:
            support = tf.cast(support, x.dtype)

            x1 = graph_propagate(x, support)

            diffusion_outputs.append(x1)

            # ----------------------------------------
            # Higher diffusion orders
            # ----------------------------------------

            xk = x1

            for _ in range(2, self.diffusion_order + 1):
                xk = graph_propagate(xk, support)

                diffusion_outputs.append(xk)

        # --------------------------------------------
        # Concatenate:
        #
        # X, AX, A²X, ...
        # --------------------------------------------

        h = tf.concat(diffusion_outputs, axis=-1)

        # --------------------------------------------
        # Equivalent role to 1x1 projection
        # --------------------------------------------

        h = self.projection(h)

        h = self.dropout(h, training=training)

        return h

In [7]:
# ============================================================
# TEST FIXED DIFFUSION GRAPH CONV
# ============================================================

fixed_support_tensors = [
    tf.constant(A_forward, dtype=tf.float32),
    tf.constant(A_reverse, dtype=tf.float32),
]


test_graph_conv = DiffusionGraphConv(
    output_channels=GW_RESIDUAL_CHANNELS, diffusion_order=GW_DIFFUSION_ORDER, dropout=GW_DROPOUT
)


test_input = tf.convert_to_tensor(X_train[:2], dtype=tf.float32)


test_output = test_graph_conv(test_input, fixed_support_tensors, training=False)


print("=== DIFFUSION GRAPH CONV TEST ===")

print("Input :", test_input.shape)

print("Supports:", len(fixed_support_tensors))

print("Output:", test_output.shape)

print("Params:", test_graph_conv.count_params())

print("All finite:", bool(tf.reduce_all(tf.math.is_finite(test_output))))

=== DIFFUSION GRAPH CONV TEST ===
Input : (2, 12, 64, 1)
Supports: 2
Output: (2, 12, 64, 32)
Params: 192
All finite: True


## 7. Adaptive Adjacency

### Tujuan

Membuat adjacency matrix yang dipelajari langsung dari data.

Graph WaveNet membentuk adaptive adjacency menggunakan dua trainable node embeddings:

- `E1`
- `E2`

Kemudian adjacency dihitung dengan:

`Softmax(ReLU(E1 × E2))`

Karakteristik:

- ukuran adjacency: `64 × 64`
- setiap baris berjumlah 1
- nilainya non-negatif
- graph dapat bersifat asimetris

In [8]:
# ============================================================
# ADAPTIVE ADJACENCY
# ============================================================


class AdaptiveAdjacency(tf.keras.layers.Layer):
    def __init__(self, num_nodes, embed_dim=10, **kwargs):

        super().__init__(**kwargs)

        self.num_nodes = num_nodes
        self.embed_dim = embed_dim

    def build(self, input_shape):

        self.nodevec1 = self.add_weight(
            name="nodevec1",
            shape=(self.num_nodes, self.embed_dim),
            initializer="random_normal",
            trainable=True,
        )

        self.nodevec2 = self.add_weight(
            name="nodevec2",
            shape=(self.embed_dim, self.num_nodes),
            initializer="random_normal",
            trainable=True,
        )

        super().build(input_shape)

    def call(self, inputs=None):

        logits = tf.matmul(self.nodevec1, self.nodevec2)

        logits = tf.nn.relu(logits)

        adjacency = tf.nn.softmax(logits, axis=1)

        return adjacency

In [9]:
# ============================================================
# TEST ADAPTIVE ADJACENCY
# ============================================================

adaptive_graph = AdaptiveAdjacency(num_nodes=NUM_NODES, embed_dim=GW_ADAPTIVE_EMBED_DIM)

A_adaptive_test = adaptive_graph(tf.zeros((1, 1), dtype=tf.float32))

print("=== ADAPTIVE ADJACENCY TEST ===")

print("Shape:", A_adaptive_test.shape)

print("Params:", adaptive_graph.count_params())

print("Min:", float(tf.reduce_min(A_adaptive_test)))

print("Max:", float(tf.reduce_max(A_adaptive_test)))

print("First 10 row sums:")

print(np.round(A_adaptive_test.numpy().sum(axis=1)[:10], 4))

print("All finite:", bool(tf.reduce_all(tf.math.is_finite(A_adaptive_test))))

print("Symmetric:", bool(np.allclose(A_adaptive_test.numpy(), A_adaptive_test.numpy().T)))

=== ADAPTIVE ADJACENCY TEST ===
Shape: (64, 64)
Params: 1280
Min: 0.015540385618805885
Max: 0.016184821724891663
First 10 row sums:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
All finite: True
Symmetric: False


## 8. Gated Dilated Temporal Convolution

### Tujuan

Membuat komponen temporal utama Graph WaveNet.

Setiap temporal layer menggunakan dua convolution:

1. filter branch dengan aktivasi `tanh`
2. gate branch dengan aktivasi `sigmoid`

Output dihitung sebagai:

`tanh(filter) × sigmoid(gate)`

Dilated convolution digunakan untuk memperluas temporal receptive field tanpa membutuhkan kernel yang besar.

In [10]:
# ============================================================
# GATED DILATED TEMPORAL CONVOLUTION
# ============================================================


class GatedTemporalConv(tf.keras.layers.Layer):
    def __init__(self, channels, kernel_size=2, dilation_rate=1, **kwargs):

        super().__init__(**kwargs)

        self.channels = channels
        self.kernel_size = kernel_size
        self.dilation_rate = dilation_rate

        self.filter_conv = tf.keras.layers.Conv2D(
            filters=channels,
            kernel_size=(kernel_size, 1),
            dilation_rate=(dilation_rate, 1),
            padding="valid",
        )

        self.gate_conv = tf.keras.layers.Conv2D(
            filters=channels,
            kernel_size=(kernel_size, 1),
            dilation_rate=(dilation_rate, 1),
            padding="valid",
        )

    def call(self, x):

        filter_out = self.filter_conv(x)

        filter_out = tf.nn.tanh(filter_out)

        gate_out = self.gate_conv(x)

        gate_out = tf.nn.sigmoid(gate_out)

        return filter_out * gate_out

In [11]:
# ============================================================
# TEST TEMPORAL CONV
# ============================================================

test_temporal = GatedTemporalConv(
    channels=GW_DILATION_CHANNELS, kernel_size=GW_KERNEL_SIZE, dilation_rate=1
)


# Graph WaveNet internal format here:
# (batch, time, nodes, channels)

test_temporal_input = tf.random.normal((2, GW_RECEPTIVE_FIELD, NUM_NODES, GW_RESIDUAL_CHANNELS))


test_temporal_output = test_temporal(test_temporal_input)


print("=== GATED TEMPORAL CONV TEST ===")

print("Input :", test_temporal_input.shape)

print("Output:", test_temporal_output.shape)

print("Params:", test_temporal.count_params())

print("All finite:", bool(tf.reduce_all(tf.math.is_finite(test_temporal_output))))

=== GATED TEMPORAL CONV TEST ===
Input : (2, 13, 64, 32)
Output: (2, 12, 64, 32)
Params: 4160
All finite: True


## 9. Graph WaveNet Layer

### Tujuan

Membuat satu layer utama Graph WaveNet.

Alur pada setiap layer:

1. gated dilated temporal convolution
2. diffusion graph convolution
3. residual connection
4. skip connection
5. batch normalization

Residual connection menjaga informasi dari layer sebelumnya.

Skip connection mengumpulkan representasi dari setiap layer untuk digunakan pada output head.

In [12]:
# ============================================================
# GRAPH WAVENET SINGLE LAYER
# ============================================================


class GraphWaveNetLayer(tf.keras.layers.Layer):
    def __init__(
        self,
        residual_channels,
        dilation_channels,
        skip_channels,
        kernel_size,
        dilation_rate,
        diffusion_order,
        dropout,
        **kwargs,
    ):

        super().__init__(**kwargs)

        self.residual_channels = residual_channels
        self.dilation_channels = dilation_channels
        self.skip_channels = skip_channels

        self.temporal = GatedTemporalConv(
            channels=dilation_channels, kernel_size=kernel_size, dilation_rate=dilation_rate
        )

        self.graph_conv = DiffusionGraphConv(
            output_channels=residual_channels, diffusion_order=diffusion_order, dropout=dropout
        )

        self.skip_conv = tf.keras.layers.Conv2D(
            filters=skip_channels, kernel_size=(1, 1), padding="valid"
        )

        self.batch_norm = tf.keras.layers.BatchNormalization()

    def call(self, x, supports, skip=None, training=False):

        # ====================================================
        # RESIDUAL INPUT
        # ====================================================

        residual = x

        # ====================================================
        # GATED DILATED TEMPORAL CONVOLUTION
        # ====================================================

        x = self.temporal(x)

        # ====================================================
        # SKIP CONNECTION
        # ====================================================

        s = self.skip_conv(x)

        if skip is None:
            skip = s

        else:
            # match temporal length
            skip = skip[:, -tf.shape(s)[1] :, :, :]

            skip = skip + s

        # ====================================================
        # GRAPH CONVOLUTION
        # ====================================================

        x = self.graph_conv(x, supports, training=training)

        # ====================================================
        # RESIDUAL CONNECTION
        # ====================================================

        residual = residual[:, -tf.shape(x)[1] :, :, :]

        x = x + residual

        # ====================================================
        # BATCH NORMALIZATION
        # ====================================================

        x = self.batch_norm(x, training=training)

        return x, skip

In [13]:
# ============================================================
# TEST ONE GRAPH WAVENET LAYER
# ============================================================

test_gw_layer = GraphWaveNetLayer(
    residual_channels=GW_RESIDUAL_CHANNELS,
    dilation_channels=GW_DILATION_CHANNELS,
    skip_channels=GW_SKIP_CHANNELS,
    kernel_size=GW_KERNEL_SIZE,
    dilation_rate=1,
    diffusion_order=GW_DIFFUSION_ORDER,
    dropout=GW_DROPOUT,
)


test_layer_input = tf.random.normal((2, GW_RECEPTIVE_FIELD, NUM_NODES, GW_RESIDUAL_CHANNELS))


test_layer_output, test_skip = test_gw_layer(
    test_layer_input, fixed_support_tensors, skip=None, training=False
)


print("=== GRAPH WAVENET LAYER TEST ===")

print("Input :", test_layer_input.shape)

print("Output:", test_layer_output.shape)

print("Skip  :", test_skip.shape)

print("Params:", test_gw_layer.count_params())

print("Output finite:", bool(tf.reduce_all(tf.math.is_finite(test_layer_output))))

print("Skip finite:", bool(tf.reduce_all(tf.math.is_finite(test_skip))))

=== GRAPH WAVENET LAYER TEST ===
Input : (2, 13, 64, 32)
Output: (2, 12, 64, 32)
Skip  : (2, 12, 64, 256)
Params: 17888
Output finite: True
Skip finite: True


## 10. Full Graph WaveNet Backbone

### Tujuan

Menyusun delapan Graph WaveNet layer sesuai konfigurasi:

- 4 blocks
- 2 layers per block
- dilation pattern `[1, 2]` pada setiap block

Sebelum masuk ke backbone, input diproyeksikan dari 1 fitur menjadi 32 residual channels.

Karena receptive field model adalah 13 sedangkan input Final64 memiliki 12 timestep, satu timestep nol ditambahkan di sisi kiri.

Skip connection dari seluruh layer kemudian dikumpulkan untuk digunakan oleh output head.

In [14]:
# ============================================================
# FULL GRAPH WAVENET BACKBONE
# ============================================================


class GraphWaveNetBackbone(tf.keras.layers.Layer):
    def __init__(
        self,
        residual_channels=32,
        dilation_channels=32,
        skip_channels=256,
        blocks=4,
        layers_per_block=2,
        kernel_size=2,
        diffusion_order=2,
        dropout=0.3,
        receptive_field=13,
        **kwargs,
    ):

        super().__init__(**kwargs)

        self.residual_channels = residual_channels
        self.skip_channels = skip_channels
        self.receptive_field = receptive_field

        # ====================================================
        # START 1x1 PROJECTION
        # ====================================================

        self.start_conv = tf.keras.layers.Conv2D(
            filters=residual_channels, kernel_size=(1, 1), padding="valid"
        )

        # ====================================================
        # GRAPH WAVENET LAYERS
        # ====================================================

        self.gw_layers = []

        for block_idx in range(blocks):
            dilation = 1

            for layer_idx in range(layers_per_block):
                layer = GraphWaveNetLayer(
                    residual_channels=residual_channels,
                    dilation_channels=dilation_channels,
                    skip_channels=skip_channels,
                    kernel_size=kernel_size,
                    dilation_rate=dilation,
                    diffusion_order=diffusion_order,
                    dropout=dropout,
                    name=(f"gw_block_{block_idx + 1}_layer_{layer_idx + 1}"),
                )

                self.gw_layers.append(layer)

                dilation *= 2

    def call(self, x, supports, training=False):

        # ====================================================
        # LEFT PAD IF INPUT < RECEPTIVE FIELD
        # ====================================================

        current_length = tf.shape(x)[1]

        pad_length = tf.maximum(self.receptive_field - current_length, 0)

        x = tf.pad(x, paddings=[[0, 0], [pad_length, 0], [0, 0], [0, 0]])

        # ====================================================
        # START PROJECTION
        # ====================================================

        x = self.start_conv(x)

        # ====================================================
        # WAVENET STACK
        # ====================================================

        skip = None

        for layer in self.gw_layers:
            x, skip = layer(x, supports, skip=skip, training=training)

        return x, skip

In [15]:
# ============================================================
# TEST FULL BACKBONE
# ============================================================

test_backbone = GraphWaveNetBackbone(
    residual_channels=GW_RESIDUAL_CHANNELS,
    dilation_channels=GW_DILATION_CHANNELS,
    skip_channels=GW_SKIP_CHANNELS,
    blocks=GW_BLOCKS,
    layers_per_block=GW_LAYERS,
    kernel_size=GW_KERNEL_SIZE,
    diffusion_order=GW_DIFFUSION_ORDER,
    dropout=GW_DROPOUT,
    receptive_field=GW_RECEPTIVE_FIELD,
)


test_backbone_input = tf.convert_to_tensor(X_train[:2], dtype=tf.float32)


test_backbone_output, test_backbone_skip = test_backbone(
    test_backbone_input, fixed_support_tensors, training=False
)


print("=== FULL BACKBONE TEST ===")

print("Original input:", test_backbone_input.shape)

print("Final residual:", test_backbone_output.shape)

print("Final skip:", test_backbone_skip.shape)

print("Layers:", len(test_backbone.gw_layers))

print("Params:", test_backbone.count_params())

print("Residual finite:", bool(tf.reduce_all(tf.math.is_finite(test_backbone_output))))

print("Skip finite:", bool(tf.reduce_all(tf.math.is_finite(test_backbone_skip))))

=== FULL BACKBONE TEST ===
Original input: (2, 12, 64, 1)
Final residual: (2, 1, 64, 32)
Final skip: (2, 1, 64, 256)
Layers: 8
Params: 143168
Residual finite: True
Skip finite: True


## 11. Output Head

### Tujuan

Mengubah skip representation terakhir menjadi prediksi untuk tiga horizon:

- t+1
- t+3
- t+6

Graph WaveNet official menggunakan dua convolution 1×1 setelah skip connection.

Pada Final64, output akhirnya disesuaikan menjadi:

`(batch, 3, 64)`

yang berarti:

- 3 horizon
- 64 node

In [16]:
# ============================================================
# GRAPH WAVENET OUTPUT HEAD
# ============================================================


class GraphWaveNetHead(tf.keras.layers.Layer):
    def __init__(self, end_channels, num_horizons, **kwargs):

        super().__init__(**kwargs)

        self.end_conv_1 = tf.keras.layers.Conv2D(
            filters=end_channels, kernel_size=(1, 1), padding="valid"
        )

        self.end_conv_2 = tf.keras.layers.Conv2D(
            filters=num_horizons, kernel_size=(1, 1), padding="valid"
        )

    def call(self, skip):

        x = tf.nn.relu(skip)

        x = self.end_conv_1(x)

        x = tf.nn.relu(x)

        x = self.end_conv_2(x)

        # current:
        # (batch, 1, nodes, horizons)

        x = tf.squeeze(x, axis=1)

        # (batch, nodes, horizons)

        x = tf.transpose(x, perm=[0, 2, 1])

        # final:
        # (batch, horizons, nodes)

        return x

In [17]:
# ============================================================
# TEST OUTPUT HEAD
# ============================================================

test_head = GraphWaveNetHead(end_channels=GW_END_CHANNELS, num_horizons=NUM_HORIZONS)


test_prediction = test_head(test_backbone_skip)


print("=== OUTPUT HEAD TEST ===")

print("Skip input :", test_backbone_skip.shape)

print("Prediction :", test_prediction.shape)

print("Params:", test_head.count_params())

print("All finite:", bool(tf.reduce_all(tf.math.is_finite(test_prediction))))

=== OUTPUT HEAD TEST ===
Skip input : (2, 1, 64, 256)
Prediction : (2, 3, 64)
Params: 133123
All finite: True


## 12. Full Graph WaveNet Model

### Tujuan

Menyatukan:

- adaptive adjacency
- fixed geographic supports
- Graph WaveNet backbone
- output head

ke dalam satu model.

Model mendukung tiga mode:

1. `fixed`
   - hanya menggunakan geographic graph

2. `adaptive`
   - hanya menggunakan learned adaptive adjacency

3. `hybrid`
   - menggunakan geographic graph dan adaptive graph secara bersamaan

Dengan desain ini, backbone temporal tetap sama sehingga perbandingan strategi graph lebih terkontrol.

In [19]:
# ============================================================
# FULL GRAPH WAVENET MODEL
# ============================================================


class GraphWaveNetModel(tf.keras.Model):
    def __init__(self, mode, fixed_supports, num_nodes, num_horizons, **kwargs):

        super().__init__(**kwargs)

        valid_modes = ["fixed", "adaptive", "hybrid"]

        if mode not in valid_modes:
            raise ValueError(f"mode harus salah satu dari {valid_modes}")

        self.mode = mode

        self.fixed_supports = [tf.constant(support, dtype=tf.float32) for support in fixed_supports]

        # ====================================================
        # ADAPTIVE GRAPH
        # ====================================================

        if mode in ["adaptive", "hybrid"]:
            self.adaptive_graph = AdaptiveAdjacency(
                num_nodes=num_nodes, embed_dim=GW_ADAPTIVE_EMBED_DIM, name="adaptive_adjacency"
            )

        else:
            self.adaptive_graph = None

        # ====================================================
        # BACKBONE
        # ====================================================

        self.backbone = GraphWaveNetBackbone(
            residual_channels=GW_RESIDUAL_CHANNELS,
            dilation_channels=GW_DILATION_CHANNELS,
            skip_channels=GW_SKIP_CHANNELS,
            blocks=GW_BLOCKS,
            layers_per_block=GW_LAYERS,
            kernel_size=GW_KERNEL_SIZE,
            diffusion_order=GW_DIFFUSION_ORDER,
            dropout=GW_DROPOUT,
            receptive_field=GW_RECEPTIVE_FIELD,
            name="graph_wavenet_backbone",
        )

        # ====================================================
        # OUTPUT HEAD
        # ====================================================

        self.head = GraphWaveNetHead(
            end_channels=GW_END_CHANNELS, num_horizons=num_horizons, name="graph_wavenet_head"
        )

    def get_supports(self):

        # ====================================================
        # FIXED
        # ====================================================

        if self.mode == "fixed":
            return self.fixed_supports

        # ====================================================
        # ADAPTIVE
        # ====================================================

        adaptive_support = self.adaptive_graph(tf.zeros((1, 1), dtype=tf.float32))

        if self.mode == "adaptive":
            return [adaptive_support]

        # ====================================================
        # HYBRID
        # ====================================================

        return self.fixed_supports + [adaptive_support]

    def call(self, x, training=False):

        supports = self.get_supports()

        _, skip = self.backbone(x, supports, training=training)

        prediction = self.head(skip)

        return prediction

In [20]:
# ============================================================
# TEST ALL THREE MODES
# ============================================================

test_models = {}

for mode in ["fixed", "adaptive", "hybrid"]:
    model = GraphWaveNetModel(
        mode=mode,
        fixed_supports=FIXED_SUPPORTS,
        num_nodes=NUM_NODES,
        num_horizons=NUM_HORIZONS,
        name=f"graph_wavenet_{mode}",
    )

    output = model(tf.convert_to_tensor(X_train[:2], dtype=tf.float32), training=False)

    test_models[mode] = model

    print(f"\n=== {mode.upper()} ===")

    print("Output:", output.shape)

    print("Params:", model.count_params())

    print("Supports:", len(model.get_supports()))

    print("Finite:", bool(tf.reduce_all(tf.math.is_finite(output))))


=== FIXED ===
Output: (2, 3, 64)
Params: 276291
Supports: 2
Finite: True

=== ADAPTIVE ===
Output: (2, 3, 64)
Params: 261187
Supports: 1
Finite: True

=== HYBRID ===
Output: (2, 3, 64)
Params: 293955
Supports: 3
Finite: True


## 13. Train Graph WaveNet Fixed — Seed 42

### Tujuan

Melatih Graph WaveNet dengan hanya menggunakan fixed geographic graph Final64.

### Konfigurasi

- Seed: 42
- Optimizer: Adam
- Learning rate: 0.001
- Batch size: 32
- Maximum epoch: 300
- Early stopping patience: 25
- Reduce LR patience: 8

Model terbaik dipilih berdasarkan validation loss.

In [21]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# BUILD FIXED MODEL
# ============================================================

tf.keras.backend.clear_session()

fixed_model = GraphWaveNetModel(
    mode="fixed",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_fixed",
)


# Build model before summary / training
_ = fixed_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)


# ============================================================
# COMPILE
# ============================================================

fixed_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)


print("=== GRAPH WAVENET FIXED ===")

print("Seed:", SEED)

print("Parameters:", fixed_model.count_params())

print("Learning rate:", GW_LEARNING_RATE)

print("Batch size:", GW_BATCH_SIZE)

print("Max epochs:", GW_MAX_EPOCHS)

=== GRAPH WAVENET FIXED ===
Seed: 42
Parameters: 276291
Learning rate: 0.001
Batch size: 32
Max epochs: 300


## 14. Train and Save Graph WaveNet Fixed

### Tujuan

Melatih Graph WaveNet Fixed seed 42 dan menyimpan:

- best model weights
- training history
- hasil evaluasi test

Best model dipilih berdasarkan `val_loss`.

In [22]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

SCREENING_DIR = ROOT / "outputs" / "final_64" / "graphwavenet_audited"

SCREENING_DIR.mkdir(parents=True, exist_ok=True)


FIXED_WEIGHTS_FILE = SCREENING_DIR / "graphwavenet_fixed_seed42.weights.h5"

FIXED_HISTORY_FILE = SCREENING_DIR / "graphwavenet_fixed_seed42_history.csv"


# ============================================================
# CALLBACKS
# ============================================================

fixed_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FIXED_WEIGHTS_FILE),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]


# ============================================================
# TRAIN
# ============================================================

fixed_history = fixed_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=fixed_callbacks,
    verbose=1,
)


# ============================================================
# SAVE HISTORY
# ============================================================

fixed_history_df = pd.DataFrame(fixed_history.history)

fixed_history_df.to_csv(FIXED_HISTORY_FILE, index=False)


# ============================================================
# SUMMARY
# ============================================================

best_epoch_fixed = fixed_history_df["val_loss"].idxmin() + 1

best_val_fixed = fixed_history_df["val_loss"].min()


print("\n=== FIXED TRAINING DONE ===")

print("Epochs run:", len(fixed_history_df))

print("Best epoch:", best_epoch_fixed)

print("Best val_loss:", best_val_fixed)

print("Weights saved:", FIXED_WEIGHTS_FILE)

print("History saved:", FIXED_HISTORY_FILE)

Epoch 1/300


E0000 00:00:1789973280.032717  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_fixed/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'graph_wave

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 0.4109

E0000 00:00:1789973285.715442  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



Epoch 1: val_loss improved from None to 0.50870, saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42.weights.h5

Epoch 1: finished saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42.weights.h5
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 144ms/step - loss: 0.4109 - val_loss: 0.5087 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 0.2539
Epoch 2: val_loss did not improve from 0.50870
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - loss: 0.2539 - val_loss: 0.5198 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 0.2313
Epoch 3: val_loss did not improve from 0.50870
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - loss: 0.2313 - val_loss: 0.5207 - learning_rate: 0.0010
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 0.2195
Epoch 4: val_loss imp

## 15. Evaluate Graph WaveNet Fixed

### Tujuan

Mengevaluasi Graph WaveNet Fixed pada test set menggunakan best weights dari training.

Metrik dihitung pada skala kecepatan asli:

- MAE
- RMSE

untuk horizon:

- t+1
- t+3
- t+6

In [23]:
# ============================================================
# LOAD BEST FIXED WEIGHTS
# ============================================================

fixed_model.load_weights(FIXED_WEIGHTS_FILE)


# ============================================================
# PREDICT TEST SET
# ============================================================

fixed_pred_std = fixed_model.predict(X_test, batch_size=GW_BATCH_SIZE, verbose=1)


# ============================================================
# EVALUATE
# ============================================================

fixed_test_results = evaluate_predictions(
    y_true_std=y_test,
    y_pred_std=fixed_pred_std,
    mask=mask_test,
    model_name="Graph WaveNet Fixed Audited",
)


print("\n=== FIXED TEST RESULTS ===")

display(fixed_test_results)


# ============================================================
# SAVE RESULTS
# ============================================================

FIXED_RESULTS_FILE = SCREENING_DIR / "graphwavenet_fixed_seed42_results.csv"

fixed_test_results.to_csv(FIXED_RESULTS_FILE, index=False)


print("\nSaved:", FIXED_RESULTS_FILE)

E0000 00:00:1789973610.940813  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 191ms/step

=== FIXED TEST RESULTS ===


,model,horizon,MAE,RMSE,N
0,Graph WaveNet Fixed Audited,t+1,2.497249,3.969238,2944
1,Graph WaveNet Fixed Audited,t+3,2.580379,4.061025,2944
2,Graph WaveNet Fixed Audited,t+6,2.573513,4.135124,3008



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_fixed_seed42_results.csv


## 16. Graph WaveNet Adaptive — Seed 42

### Tujuan

Membangun Graph WaveNet dengan backbone yang sama seperti model Fixed.

Perbedaannya hanya pada graph support:

- Fixed menggunakan dua geographic transition matrices.
- Adaptive menggunakan satu adjacency matrix yang dipelajari selama training.

Adaptive adjacency dibentuk dari trainable node embeddings menggunakan:

`Softmax(ReLU(E1 × E2))`

Eksperimen menggunakan seed 42 agar konsisten dengan Graph WaveNet Fixed.

In [24]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# CLEAR PREVIOUS GRAPH
# ============================================================

tf.keras.backend.clear_session()


# ============================================================
# BUILD ADAPTIVE MODEL
# ============================================================

adaptive_model = GraphWaveNetModel(
    mode="adaptive",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_adaptive",
)


# Build model
_ = adaptive_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)


# ============================================================
# COMPILE
# ============================================================

adaptive_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)


# ============================================================
# CHECK
# ============================================================

print("=== GRAPH WAVENET ADAPTIVE ===")

print("Seed:", SEED)

print("Parameters:", adaptive_model.count_params())

print("Supports:", len(adaptive_model.get_supports()))

print("Learning rate:", GW_LEARNING_RATE)

print("Batch size:", GW_BATCH_SIZE)

print("Max epochs:", GW_MAX_EPOCHS)


# ============================================================
# ADAPTIVE GRAPH INITIAL CHECK
# ============================================================

initial_adaptive_adj = adaptive_model.adaptive_graph(tf.zeros((1, 1), dtype=tf.float32)).numpy()


print("\nAdaptive adjacency:", initial_adaptive_adj.shape)

print("Row sums first 5:", np.round(initial_adaptive_adj.sum(axis=1)[:5], 4))

print("Min:", initial_adaptive_adj.min())

print("Max:", initial_adaptive_adj.max())

=== GRAPH WAVENET ADAPTIVE ===
Seed: 42
Parameters: 261187
Supports: 1
Learning rate: 0.001
Batch size: 32
Max epochs: 300

Adaptive adjacency: (64, 64)
Row sums first 5: [1. 1. 1. 1. 1.]
Min: 0.015536633
Max: 0.01609368


## 17. Train and Save Graph WaveNet Adaptive

### Tujuan

Melatih Graph WaveNet Adaptive seed 42 dan menyimpan:

- best model weights
- training history
- learned adaptive adjacency setelah training

Best model dipilih berdasarkan validation loss.

In [25]:
# ============================================================
# OUTPUT FILES
# ============================================================

ADAPTIVE_WEIGHTS_FILE = SCREENING_DIR / "graphwavenet_adaptive_seed42.weights.h5"

ADAPTIVE_HISTORY_FILE = SCREENING_DIR / "graphwavenet_adaptive_seed42_history.csv"

ADAPTIVE_GRAPH_FILE = SCREENING_DIR / "graphwavenet_adaptive_seed42_adjacency.npy"


# ============================================================
# CALLBACKS
# ============================================================

adaptive_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ADAPTIVE_WEIGHTS_FILE),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]


# ============================================================
# TRAIN
# ============================================================

adaptive_history = adaptive_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=adaptive_callbacks,
    verbose=1,
)


# ============================================================
# SAVE HISTORY
# ============================================================

adaptive_history_df = pd.DataFrame(adaptive_history.history)

adaptive_history_df.to_csv(ADAPTIVE_HISTORY_FILE, index=False)


# ============================================================
# BEST EPOCH
# ============================================================

best_epoch_adaptive = adaptive_history_df["val_loss"].idxmin() + 1

best_val_adaptive = adaptive_history_df["val_loss"].min()


# ============================================================
# LOAD BEST WEIGHTS
# ============================================================

adaptive_model.load_weights(ADAPTIVE_WEIGHTS_FILE)


# ============================================================
# SAVE BEST ADAPTIVE GRAPH
# ============================================================

learned_adaptive_adj = adaptive_model.adaptive_graph(tf.zeros((1, 1), dtype=tf.float32)).numpy()

np.save(ADAPTIVE_GRAPH_FILE, learned_adaptive_adj)


# ============================================================
# SUMMARY
# ============================================================

print("\n=== ADAPTIVE TRAINING DONE ===")

print("Epochs run:", len(adaptive_history_df))

print("Best epoch:", best_epoch_adaptive)

print("Best val_loss:", best_val_adaptive)

print("Weights saved:", ADAPTIVE_WEIGHTS_FILE)

print("History saved:", ADAPTIVE_HISTORY_FILE)

print("Adaptive graph saved:", ADAPTIVE_GRAPH_FILE)

Epoch 1/300


E0000 00:00:1789973683.967535  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_adaptive/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'graph_w

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 0.3984

E0000 00:00:1789973689.618445  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



Epoch 1: val_loss improved from None to 0.54501, saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42.weights.h5

Epoch 1: finished saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42.weights.h5
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 132ms/step - loss: 0.3984 - val_loss: 0.5450 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - loss: 0.2486
Epoch 2: val_loss did not improve from 0.54501
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - loss: 0.2486 - val_loss: 0.5572 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 0.2326
Epoch 3: val_loss improved from 0.54501 to 0.52572, saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42.weights.h5

Epoch 3: fin

## 18. Evaluate Graph WaveNet Adaptive

### Tujuan

Mengevaluasi Graph WaveNet Adaptive menggunakan best weights.

Metrik dihitung pada skala kecepatan asli:

- MAE
- RMSE

untuk:

- t+1
- t+3
- t+6

In [26]:
# ============================================================
# LOAD BEST ADAPTIVE WEIGHTS
# ============================================================

adaptive_model.load_weights(ADAPTIVE_WEIGHTS_FILE)


# ============================================================
# PREDICT TEST SET
# ============================================================

adaptive_pred_std = adaptive_model.predict(X_test, batch_size=GW_BATCH_SIZE, verbose=1)


# ============================================================
# EVALUATE
# ============================================================

adaptive_test_results = evaluate_predictions(
    y_true_std=y_test,
    y_pred_std=adaptive_pred_std,
    mask=mask_test,
    model_name="Graph WaveNet Adaptive Audited",
)


print("\n=== ADAPTIVE TEST RESULTS ===")

display(adaptive_test_results)


# ============================================================
# SAVE RESULTS
# ============================================================

ADAPTIVE_RESULTS_FILE = SCREENING_DIR / "graphwavenet_adaptive_seed42_results.csv"

adaptive_test_results.to_csv(ADAPTIVE_RESULTS_FILE, index=False)


print("\nSaved:", ADAPTIVE_RESULTS_FILE)

E0000 00:00:1789973873.092580  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 216ms/step

=== ADAPTIVE TEST RESULTS ===


,model,horizon,MAE,RMSE,N
0,Graph WaveNet Adaptive Audited,t+1,2.282072,3.836623,2944
1,Graph WaveNet Adaptive Audited,t+3,2.501486,4.026375,2944
2,Graph WaveNet Adaptive Audited,t+6,2.538214,4.022828,3008



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_adaptive_seed42_results.csv


## 19. Fixed vs Adaptive Comparison

### Tujuan

Membandingkan Graph WaveNet Fixed dan Adaptive pada seed 42.

Improvement dihitung sebagai pengurangan error dari Fixed ke Adaptive:

`Improvement (%) = (Fixed - Adaptive) / Fixed × 100`

Nilai positif berarti Adaptive menghasilkan error yang lebih rendah.

In [27]:
# ============================================================
# FIXED VS ADAPTIVE COMPARISON
# ============================================================

comparison_rows = []

for i, horizon in enumerate(HORIZONS):
    fixed_row = fixed_test_results.iloc[i]
    adaptive_row = adaptive_test_results.iloc[i]

    mae_improvement = (fixed_row["MAE"] - adaptive_row["MAE"]) / fixed_row["MAE"] * 100

    rmse_improvement = (fixed_row["RMSE"] - adaptive_row["RMSE"]) / fixed_row["RMSE"] * 100

    comparison_rows.append(
        {
            "horizon": f"t+{horizon}",
            "fixed_MAE": fixed_row["MAE"],
            "adaptive_MAE": adaptive_row["MAE"],
            "MAE_improvement_%": mae_improvement,
            "fixed_RMSE": fixed_row["RMSE"],
            "adaptive_RMSE": adaptive_row["RMSE"],
            "RMSE_improvement_%": rmse_improvement,
        }
    )


fixed_adaptive_comparison = pd.DataFrame(comparison_rows)


print("=== FIXED VS ADAPTIVE ===")

display(fixed_adaptive_comparison)


# ============================================================
# SAVE
# ============================================================

COMPARISON_FILE = SCREENING_DIR / "graphwavenet_fixed_vs_adaptive_seed42.csv"

fixed_adaptive_comparison.to_csv(COMPARISON_FILE, index=False)


print("\nSaved:", COMPARISON_FILE)

=== FIXED VS ADAPTIVE ===


,horizon,fixed_MAE,adaptive_MAE,MAE_improvement_%,fixed_RMSE,adaptive_RMSE,RMSE_improvement_%
0,t+1,2.497249,2.282072,8.616564,3.969238,3.836623,3.341063
1,t+3,2.580379,2.501486,3.057426,4.061025,4.026375,0.853225
2,t+6,2.573513,2.538214,1.371611,4.135124,4.022828,2.715662



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_fixed_vs_adaptive_seed42.csv


## 20. Graph WaveNet Hybrid — Seed 42

### Tujuan

Membangun Graph WaveNet Hybrid yang menggunakan:

- fixed geographic supports
- learned adaptive adjacency

secara bersamaan.

Backbone temporal tetap sama seperti model Fixed dan Adaptive.

Eksperimen ini digunakan sebagai ablation untuk melihat apakah kombinasi geographic graph dan adaptive graph memberikan tambahan manfaat.

In [28]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# CLEAR SESSION
# ============================================================

tf.keras.backend.clear_session()


# ============================================================
# BUILD HYBRID MODEL
# ============================================================

hybrid_model = GraphWaveNetModel(
    mode="hybrid",
    fixed_supports=FIXED_SUPPORTS,
    num_nodes=NUM_NODES,
    num_horizons=NUM_HORIZONS,
    name="graph_wavenet_hybrid",
)


# Build model
_ = hybrid_model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)


# ============================================================
# COMPILE
# ============================================================

hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
)


# ============================================================
# CHECK
# ============================================================

print("=== GRAPH WAVENET HYBRID ===")

print("Seed:", SEED)

print("Parameters:", hybrid_model.count_params())

print("Supports:", len(hybrid_model.get_supports()))

print("Learning rate:", GW_LEARNING_RATE)

print("Batch size:", GW_BATCH_SIZE)

print("Max epochs:", GW_MAX_EPOCHS)

=== GRAPH WAVENET HYBRID ===
Seed: 42
Parameters: 293955
Supports: 3
Learning rate: 0.001
Batch size: 32
Max epochs: 300


## 21. Train and Save Graph WaveNet Hybrid

### Tujuan

Melatih Graph WaveNet Hybrid seed 42.

Model menggunakan tiga graph supports:

- forward geographic support
- reverse geographic support
- learned adaptive support

Best weights, training history, dan adaptive adjacency dari model Hybrid disimpan untuk analisis berikutnya.

In [29]:
# ============================================================
# OUTPUT FILES
# ============================================================

HYBRID_WEIGHTS_FILE = SCREENING_DIR / "graphwavenet_hybrid_seed42.weights.h5"

HYBRID_HISTORY_FILE = SCREENING_DIR / "graphwavenet_hybrid_seed42_history.csv"

HYBRID_GRAPH_FILE = SCREENING_DIR / "graphwavenet_hybrid_seed42_adjacency.npy"


# ============================================================
# CALLBACKS
# ============================================================

hybrid_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(HYBRID_WEIGHTS_FILE),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]


# ============================================================
# TRAIN
# ============================================================

hybrid_history = hybrid_model.fit(
    X_train,
    y_train_masked,
    validation_data=(X_val, y_val_masked),
    epochs=GW_MAX_EPOCHS,
    batch_size=GW_BATCH_SIZE,
    callbacks=hybrid_callbacks,
    verbose=1,
)


# ============================================================
# SAVE HISTORY
# ============================================================

hybrid_history_df = pd.DataFrame(hybrid_history.history)

hybrid_history_df.to_csv(HYBRID_HISTORY_FILE, index=False)


# ============================================================
# BEST EPOCH
# ============================================================

best_epoch_hybrid = hybrid_history_df["val_loss"].idxmin() + 1

best_val_hybrid = hybrid_history_df["val_loss"].min()


# ============================================================
# LOAD BEST WEIGHTS
# ============================================================

hybrid_model.load_weights(HYBRID_WEIGHTS_FILE)


# ============================================================
# SAVE HYBRID ADAPTIVE GRAPH
# ============================================================

hybrid_adaptive_adj = hybrid_model.adaptive_graph(tf.zeros((1, 1), dtype=tf.float32)).numpy()

np.save(HYBRID_GRAPH_FILE, hybrid_adaptive_adj)


# ============================================================
# SUMMARY
# ============================================================

print("\n=== HYBRID TRAINING DONE ===")

print("Epochs run:", len(hybrid_history_df))

print("Best epoch:", best_epoch_hybrid)

print("Best val_loss:", best_val_hybrid)

print("Weights saved:", HYBRID_WEIGHTS_FILE)

print("History saved:", HYBRID_HISTORY_FILE)

print("Adaptive graph saved:", HYBRID_GRAPH_FILE)

Epoch 1/300


E0000 00:00:1789974035.025879  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graph_wavenet_hybrid/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'graph_wav

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 0.3636

E0000 00:00:1789974041.065898  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



Epoch 1: val_loss improved from None to 0.45841, saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_hybrid_seed42.weights.h5

Epoch 1: finished saving model to /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_hybrid_seed42.weights.h5
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 161ms/step - loss: 0.3636 - val_loss: 0.4584 - learning_rate: 0.0010
Epoch 2/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 0.2351
Epoch 2: val_loss did not improve from 0.45841
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - loss: 0.2351 - val_loss: 0.4669 - learning_rate: 0.0010
Epoch 3/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 0.2151
Epoch 3: val_loss did not improve from 0.45841
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 131ms/step - loss: 0.2151 - val_loss: 0.4770 - learning_rate: 0.0010
Epoch 4/300
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - loss: 0.2042
Epoch 4: val_los

## 22. Evaluate Graph WaveNet Hybrid

### Tujuan

Mengevaluasi Graph WaveNet Hybrid menggunakan best weights.

Metrik dihitung pada skala kecepatan asli:

- MAE
- RMSE

untuk:

- t+1
- t+3
- t+6

In [30]:
# ============================================================
# LOAD BEST HYBRID WEIGHTS
# ============================================================

hybrid_model.load_weights(HYBRID_WEIGHTS_FILE)


# ============================================================
# PREDICT TEST SET
# ============================================================

hybrid_pred_std = hybrid_model.predict(X_test, batch_size=GW_BATCH_SIZE, verbose=1)


# ============================================================
# EVALUATE
# ============================================================

hybrid_test_results = evaluate_predictions(
    y_true_std=y_test,
    y_pred_std=hybrid_pred_std,
    mask=mask_test,
    model_name="Graph WaveNet Hybrid Audited",
)


print("\n=== HYBRID TEST RESULTS ===")

display(hybrid_test_results)


# ============================================================
# SAVE RESULTS
# ============================================================

HYBRID_RESULTS_FILE = SCREENING_DIR / "graphwavenet_hybrid_seed42_results.csv"

hybrid_test_results.to_csv(HYBRID_RESULTS_FILE, index=False)


print("\nSaved:", HYBRID_RESULTS_FILE)

E0000 00:00:1789974283.516926  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/stepWARNING:tensorflow:6 out of the last 9 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x129813e20> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 348ms/step

=== HYBRID TEST RESULTS ===


,model,horizon,MAE,RMSE,N
0,Graph WaveNet Hybrid Audited,t+1,2.223365,3.762612,2944
1,Graph WaveNet Hybrid Audited,t+3,2.334818,3.879791,2944
2,Graph WaveNet Hybrid Audited,t+6,2.357840,3.785689,3008



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_hybrid_seed42_results.csv


## 23. Final Audited Graph WaveNet Comparison — Seed 42

### Tujuan

Membandingkan tiga konfigurasi Graph WaveNet:

- Fixed
- Adaptive
- Hybrid

Semua menggunakan dataset, target, mask, dan backbone temporal yang sama.

Perbedaan utama berada pada graph supports yang digunakan.

In [31]:
# ============================================================
# FINAL SEED 42 COMPARISON
# ============================================================

final_seed42_results = pd.concat(
    [fixed_test_results, adaptive_test_results, hybrid_test_results], ignore_index=True
)


final_seed42_table = final_seed42_results.pivot(
    index="model", columns="horizon", values=["MAE", "RMSE"]
)


# Flatten column names
final_seed42_table.columns = [
    f"{metric} {horizon}" for metric, horizon in final_seed42_table.columns
]


# Reorder columns
final_seed42_table = final_seed42_table[
    ["MAE t+1", "MAE t+3", "MAE t+6", "RMSE t+1", "RMSE t+3", "RMSE t+6"]
]


print("=== FINAL AUDITED GRAPH WAVENET — SEED 42 ===")

display(final_seed42_table)


# ============================================================
# SAVE
# ============================================================

FINAL_SEED42_FILE = SCREENING_DIR / "graphwavenet_audited_seed42_comparison.csv"

final_seed42_table.to_csv(FINAL_SEED42_FILE)


print("\nSaved:", FINAL_SEED42_FILE)

=== FINAL AUDITED GRAPH WAVENET — SEED 42 ===


,MAE t+1,MAE t+3,MAE t+6,RMSE t+1,RMSE t+3,RMSE t+6
model,,,,,,
Graph WaveNet Adaptive Audited,2.282072,2.501486,2.538214,3.836623,4.026375,4.022828
Graph WaveNet Fixed Audited,2.497249,2.580379,2.573513,3.969238,4.061025,4.135124
Graph WaveNet Hybrid Audited,2.223365,2.334818,2.357840,3.762612,3.879791,3.785689



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_audited_seed42_comparison.csv


## 24. Inspect Learned Adaptive Graph

### Tujuan

Memeriksa apakah adaptive adjacency yang dipelajari setelah training memiliki struktur yang bermakna dan tidak tetap hampir uniform.

Pemeriksaan dilakukan pada:

- Adaptive model
- Hybrid model

Yang dicek:

- minimum dan maksimum weight
- distribusi weight
- entropy rata-rata per row
- top-k neighbor weight per node
- perbedaan terhadap uniform adjacency

In [32]:
# ============================================================
# LOAD LEARNED ADJACENCIES
# ============================================================

adaptive_adj = np.load(ADAPTIVE_GRAPH_FILE)

hybrid_adj = np.load(HYBRID_GRAPH_FILE)


# ============================================================
# HELPER
# ============================================================


def summarize_adjacency(adj, name):

    eps = 1e-12

    uniform_value = 1.0 / adj.shape[1]

    entropy = -np.sum(adj * np.log(adj + eps), axis=1)

    normalized_entropy = entropy / np.log(adj.shape[1])

    top1 = np.max(adj, axis=1)

    sorted_weights = np.sort(adj, axis=1)[:, ::-1]

    top5_sum = np.sum(sorted_weights[:, :5], axis=1)

    mean_abs_from_uniform = np.mean(np.abs(adj - uniform_value))

    print(f"\n=== {name} ===")

    print("Shape:", adj.shape)

    print("Min:", float(adj.min()))

    print("Max:", float(adj.max()))

    print("Mean:", float(adj.mean()))

    print("Mean top-1 weight:", float(top1.mean()))

    print("Mean top-5 mass:", float(top5_sum.mean()))

    print("Mean normalized entropy:", float(normalized_entropy.mean()))

    print("Mean absolute deviation from uniform:", float(mean_abs_from_uniform))

    print("Row sums first 5:", np.round(adj.sum(axis=1)[:5], 4))


summarize_adjacency(adaptive_adj, "Adaptive model adjacency")

summarize_adjacency(hybrid_adj, "Hybrid model adaptive adjacency")


=== Adaptive model adjacency ===
Shape: (64, 64)
Min: 0.009163577109575272
Max: 0.13315512239933014
Mean: 0.015625
Mean top-1 weight: 0.05727740004658699
Mean top-5 mass: 0.19178931415081024
Mean normalized entropy: 0.9719928045161927
Mean absolute deviation from uniform: 0.005801750347018242
Row sums first 5: [1. 1. 1. 1. 1.]

=== Hybrid model adaptive adjacency ===
Shape: (64, 64)
Min: 0.009219028055667877
Max: 0.054757002741098404
Mean: 0.015625
Mean top-1 weight: 0.038957953453063965
Mean top-5 mass: 0.15249383449554443
Mean normalized entropy: 0.982147365172853
Mean absolute deviation from uniform: 0.005056942813098431
Row sums first 5: [1. 1. 1. 1. 1.]


## 25. Multi-Seed Training Helper

### Tujuan

Menyiapkan fungsi reusable untuk eksperimen robustness Graph WaveNet.

Setiap eksperimen akan:

1. mengatur random seed
2. membuat model baru
3. melatih model menggunakan train set
4. memilih model terbaik berdasarkan validation loss
5. memuat kembali best weights
6. mengevaluasi test set
7. menyimpan:
   - weights
   - training history
   - test metrics
   - learned adaptive adjacency jika tersedia

Seed yang digunakan:

- 42
- 123
- 456
- 789
- 2026

Model:

- Fixed
- Adaptive
- Hybrid

In [33]:
# ============================================================
# MULTI-SEED CONFIG
# ============================================================

MULTI_SEEDS = [42, 123, 456, 789, 2026]

MULTISEED_DIR = SCREENING_DIR / "multiseed"

MULTISEED_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# TRAIN ONE MODEL / ONE SEED
# ============================================================


def train_gwn_seed(mode, seed, verbose=0):

    print("\n" + "=" * 65)

    print(f"Graph WaveNet {mode.upper()} | Seed {seed}")

    print("=" * 65)

    # ========================================================
    # RESET SESSION + SEED
    # ========================================================

    tf.keras.backend.clear_session()

    np.random.seed(seed)

    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

    # ========================================================
    # OUTPUT PATHS
    # ========================================================

    run_name = f"graphwavenet_{mode}_seed{seed}"

    weights_file = MULTISEED_DIR / f"{run_name}.weights.h5"

    history_file = MULTISEED_DIR / f"{run_name}_history.csv"

    results_file = MULTISEED_DIR / f"{run_name}_results.csv"

    graph_file = MULTISEED_DIR / f"{run_name}_adjacency.npy"

    # ========================================================
    # BUILD MODEL
    # ========================================================

    model = GraphWaveNetModel(
        mode=mode,
        fixed_supports=FIXED_SUPPORTS,
        num_nodes=NUM_NODES,
        num_horizons=NUM_HORIZONS,
        name=run_name,
    )

    # Build model
    _ = model(tf.convert_to_tensor(X_train[:1], dtype=tf.float32), training=False)

    # ========================================================
    # COMPILE
    # ========================================================

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=GW_LEARNING_RATE), loss=masked_mse
    )

    # ========================================================
    # CALLBACKS
    # ========================================================

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(weights_file),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=25, restore_best_weights=True, verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=0
        ),
    ]

    # ========================================================
    # TRAIN
    # ========================================================

    history = model.fit(
        X_train,
        y_train_masked,
        validation_data=(X_val, y_val_masked),
        epochs=GW_MAX_EPOCHS,
        batch_size=GW_BATCH_SIZE,
        callbacks=callbacks,
        verbose=verbose,
    )

    # ========================================================
    # HISTORY
    # ========================================================

    history_df = pd.DataFrame(history.history)

    history_df.to_csv(history_file, index=False)

    best_epoch = history_df["val_loss"].idxmin() + 1

    best_val_loss = float(history_df["val_loss"].min())

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_weights(weights_file)

    # ========================================================
    # TEST PREDICTION
    # ========================================================

    pred_std = model.predict(X_test, batch_size=GW_BATCH_SIZE, verbose=0)

    test_results = evaluate_predictions(
        y_true_std=y_test,
        y_pred_std=pred_std,
        mask=mask_test,
        model_name=f"Graph WaveNet {mode.title()}",
    )

    # ========================================================
    # ADD RUN METADATA
    # ========================================================

    test_results.insert(0, "seed", seed)

    test_results.insert(1, "mode", mode)

    test_results["best_epoch"] = best_epoch

    test_results["best_val_loss"] = best_val_loss

    test_results["params"] = model.count_params()

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    test_results.to_csv(results_file, index=False)

    # ========================================================
    # SAVE LEARNED GRAPH
    # ========================================================

    if model.adaptive_graph is not None:
        learned_adj = model.adaptive_graph(tf.zeros((1, 1), dtype=tf.float32)).numpy()

        np.save(graph_file, learned_adj)

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print("Params       :", model.count_params())

    print("Epochs run   :", len(history_df))

    print("Best epoch   :", best_epoch)

    print("Best val_loss:", best_val_loss)

    display(test_results[["horizon", "MAE", "RMSE", "N"]])

    return {"model": model, "history": history_df, "results": test_results, "predictions": pred_std}


print("Multi-seed helper ready.")

print("Seeds:", MULTI_SEEDS)

print("Output:", MULTISEED_DIR)

Multi-seed helper ready.
Seeds: [42, 123, 456, 789, 2026]
Output: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/multiseed


## 26. Run Remaining Multi-Seed Experiments

### Tujuan

Menjalankan eksperimen untuk seed:

- 123
- 456
- 789
- 2026

pada tiga mode:

- Fixed
- Adaptive
- Hybrid

Seed 42 tidak diulang karena sebelumnya sudah selesai dan hasilnya sudah disimpan.

In [34]:
# ============================================================
# RUN REMAINING MULTI-SEED EXPERIMENTS
# ============================================================

REMAINING_SEEDS = [123, 456, 789, 2026]

MULTISEED_RUNS = {}


for mode in ["fixed", "adaptive", "hybrid"]:
    MULTISEED_RUNS[mode] = {}

    for seed in REMAINING_SEEDS:
        result = train_gwn_seed(mode=mode, seed=seed, verbose=0)

        MULTISEED_RUNS[mode][seed] = result


print("\n=== ALL REMAINING RUNS FINISHED ===")

print("Modes:", list(MULTISEED_RUNS.keys()))

print("Seeds:", REMAINING_SEEDS)


Graph WaveNet FIXED | Seed 123


E0000 00:00:1789974689.554436  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graphwavenet_fixed_seed123/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'gra

Params       : 276291
Epochs run   : 82
Best epoch   : 57
Best val_loss: 0.14523065090179443


,horizon,MAE,RMSE,N
0,t+1,2.364608,4.046557,2944
1,t+3,2.507490,4.036215,2944
2,t+6,2.556224,4.097866,3008



Graph WaveNet FIXED | Seed 456


/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graphwavenet_fixed_seed456/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'graphwavenet_fixed_seed456/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/bias', 'graphwavenet_fixed_seed456/graph_wavenet_backbone/gw_block_4_layer_2/batch_normalization_7/gamma', 'graphwavenet_fixed_seed456/graph_wavenet_backbone/gw_block_4_layer_2/batch_normalization_7/beta'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(
E0000 00:00:1789974818.157890  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list

... (39 outputs left out)

/Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:862: UserWarning: Gradients do not exist for variables ['graphwavenet_hybrid_seed2026/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/kernel', 'graphwavenet_hybrid_seed2026/graph_wavenet_backbone/gw_block_4_layer_2/diffusion_graph_conv_7/dense_7/bias', 'graphwavenet_hybrid_seed2026/graph_wavenet_backbone/gw_block_4_layer_2/batch_normalization_7/gamma', 'graphwavenet_hybrid_seed2026/graph_wavenet_backbone/gw_block_4_layer_2/batch_normalization_7/beta'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(
E0000 00:00:1789977090.237271  249237 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targume

Params       : 293955
Epochs run   : 90
Best epoch   : 65
Best val_loss: 0.13512247800827026


,horizon,MAE,RMSE,N
0,t+1,2.231920,3.789765,2944
1,t+3,2.316777,3.857996,2944
2,t+6,2.424995,3.955624,3008



=== ALL REMAINING RUNS FINISHED ===
Modes: ['fixed', 'adaptive', 'hybrid']
Seeds: [123, 456, 789, 2026]


## 27. Multi-Seed Summary

### Tujuan

Menggabungkan hasil Graph WaveNet dari lima seed:

- 42
- 123
- 456
- 789
- 2026

untuk model:

- Fixed
- Adaptive
- Hybrid

Kemudian dihitung mean dan standard deviation untuk MAE dan RMSE pada horizon t+1, t+3, dan t+6.

In [35]:
# ============================================================
# LOAD ALL MULTI-SEED RESULTS
# ============================================================

all_rows = []


# seed 42 yang sebelumnya disimpan di folder utama
seed42_files = {
    "fixed": SCREENING_DIR / "graphwavenet_fixed_seed42_results.csv",
    "adaptive": SCREENING_DIR / "graphwavenet_adaptive_seed42_results.csv",
    "hybrid": SCREENING_DIR / "graphwavenet_hybrid_seed42_results.csv",
}


for mode, file_path in seed42_files.items():
    df = pd.read_csv(file_path)

    df["seed"] = 42
    df["mode"] = mode

    all_rows.append(df)


# ============================================================
# SEED 123 / 456 / 789 / 2026
# ============================================================

for mode in ["fixed", "adaptive", "hybrid"]:
    for seed in [123, 456, 789, 2026]:
        file_path = MULTISEED_DIR / f"graphwavenet_{mode}_seed{seed}_results.csv"

        df = pd.read_csv(file_path)

        all_rows.append(df)


# ============================================================
# COMBINE
# ============================================================

all_multiseed_results = pd.concat(all_rows, ignore_index=True)


print("Total rows:", len(all_multiseed_results))

display(all_multiseed_results.head())


# ============================================================
# SUMMARY MEAN + STD
# ============================================================

multiseed_summary = (
    all_multiseed_results.groupby(["mode", "horizon"])
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
    )
    .reset_index()
)


print("\n=== MULTI-SEED SUMMARY ===")

display(multiseed_summary)


# ============================================================
# SAVE
# ============================================================

ALL_RESULTS_FILE = SCREENING_DIR / "graphwavenet_multiseed_all_results.csv"

SUMMARY_FILE = SCREENING_DIR / "graphwavenet_multiseed_summary.csv"


all_multiseed_results.to_csv(ALL_RESULTS_FILE, index=False)

multiseed_summary.to_csv(SUMMARY_FILE, index=False)


print("\nSaved:", ALL_RESULTS_FILE)

print("Saved:", SUMMARY_FILE)

Total rows: 45


,model,horizon,MAE,RMSE,N,seed,mode,best_epoch,best_val_loss,params
0,Graph WaveNet Fixed Audited,t+1,2.497249,3.969238,2944,42,fixed,NaN,NaN,NaN
1,Graph WaveNet Fixed Audited,t+3,2.580379,4.061025,2944,42,fixed,NaN,NaN,NaN
2,Graph WaveNet Fixed Audited,t+6,2.573513,4.135124,3008,42,fixed,NaN,NaN,NaN
3,Graph WaveNet Adaptive Audited,t+1,2.282072,3.836623,2944,42,adaptive,NaN,NaN,NaN
4,Graph WaveNet Adaptive Audited,t+3,2.501486,4.026375,2944,42,adaptive,NaN,NaN,NaN



=== MULTI-SEED SUMMARY ===


,mode,horizon,MAE_mean,MAE_std,RMSE_mean,RMSE_std
0,adaptive,t+1,2.281376,0.042020,3.872618,0.042626
1,adaptive,t+3,2.491646,0.076727,4.068399,0.076556
2,adaptive,t+6,2.583704,0.070752,4.081930,0.083692
3,fixed,t+1,2.437737,0.107950,4.011632,0.033994
4,fixed,t+3,2.525568,0.054664,4.079538,0.060782
5,fixed,t+6,2.548207,0.047744,4.080638,0.053301
6,hybrid,t+1,2.263973,0.056467,3.797336,0.022564
7,hybrid,t+3,2.404709,0.101892,3.949592,0.098253
8,hybrid,t+6,2.426071,0.068632,3.908812,0.080742



Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_multiseed_all_results.csv
Saved: /Users/aisha/Documents/SKRIPSI 2026/SEMINAR 2026/tomtom-stgnn-agent/outputs/final_64/graphwavenet_audited/graphwavenet_multiseed_summary.csv
